# `mse_nqubits.ipynb` -- annotated

**Paper:** *Fermi-Dirac machines as quantizations of neurons* (A. He, N. Liu, M. M. Wilde).

This notebook reproduces the **squared-loss / function-approximation** experiments of **Section VI.D.1** (Figure 7). It trains a *quantized neuron* whose activation function is `tanh` -- i.e. the **Fermi-Dirac neuron** $g_T(x)=\tanh(x/T)$ (Eq. (18)) -- and compares a **quantum model** against a **classical model**.

| Code object | Paper |
|---|---|
| `generate_paulis(..., 'quantum')` | Transverse-field Ising model $H_{\mathrm{TFIM}}(\omega)$, **Eq. (113)** |
| `generate_paulis(..., 'classical')` | Classical Ising model $H_{\mathrm{IM}}(\omega)$, **Eq. (114)** |
| `H(omega) = sum_j omega_j H_j` | Parameterized Hamiltonian, **Eq. (16)** |
| `tanh(H/T)` activation | $g_T(x)=\tanh(x/T)$, **Eq. (18)**; objective $\mathrm{Tr}[g_T(H(\omega))\rho]$, **Eq. (17)** |
| `dfj` / `fdd_tanh_matrix` | Gradient of objective, **Theorem 1 / Eq. (20)**, derivative of matrix `tanh` (**Appendix A.1**) |
| `optimize` loop | Training protocol **Sec. VI.C**, steps 1-5; update **Eq. (119)** |
| training states | Data generation **Sec. VI.A**, **Eq. (110)-(111)**; target/labels **Eq. (120)** |

*Annotations are comments only; no executable code was changed.*

In [ ]:
import numpy as np
from scipy.linalg import tanhm
import matplotlib.pyplot as plt
import itertools
import pandas as pd

In [ ]:
# --- Single-qubit Pauli operators I, X, Y, Z (Paper Sec. II.A). The
#     parameterized Hamiltonian H(omega)=sum_j omega_j H_j, Eq. (16), is
#     built from tensor products ("Pauli strings") of these. ---
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# krons: Kronecker (tensor) product of a list of single-qubit ops -> one
#   n-qubit Pauli string H_j acting on the 2^n-dim Hilbert space (Sec. II.A).
def krons(ops):
    res = ops[0]
    for op in ops[1:]:
        res = np.kron(res, op)
    return res

# to_density: |psi> -> rho = |psi><psi|, a pure-state density matrix.
#   These rho are the input states "rho" fed to the neuron (Eq. (17)).
def to_density(vec):
    return np.outer(vec, vec.conj())

def generate_paulis(n, model="quantum"):
    """
    Generates Pauli strings for an n-qubit system.
    Includes nearest neightbor ZZ interactions, 1-body field terms, and Identity.

    Paper: builds the term operators {H_j} for the parameterized Hamiltonian
    H(omega)=sum_j omega_j H_j, Eq. (16).
      model="quantum"  -> TFIM, Eq. (113): ZZ couplings + transverse X field.
      model="classical"-> classical Ising model (IM), Eq. (114): ZZ + Z field.
    The ONLY difference between the two is the 1-body term (X vs Z); note that
    [Z_i Z_{i+1}, X] do not commute (quantum) while all Z terms commute
    (classical). Sec. II.A: non-commuting terms are what lift the model beyond
    a classical neuron.
    """
    paulis = []
    
    # ZZ Interactions: nearest-neighbor Z_i (x) Z_{i+1} coupling (the
    #   "-W_i Z^(i) Z^(i+1)" sum shared by TFIM Eq. (113) and IM Eq. (114)).
    for i in range(n-1):
        ops = [I] * n
        ops[i] = Z
        ops[i+1] = Z
        paulis.append(krons(ops))
            
    # 1-body field term. quantum: transverse X field -> TFIM (Eq. 113,
    #   non-commuting w/ ZZ). classical: longitudinal Z field -> IM (Eq. 114).
    for i in range(n):
        ops = [I] * n
        if model == "quantum":
            ops[i] = X
        elif model == "classical":
            ops[i] = Z
        paulis.append(krons(ops))
        
    # Identity term b*I (the bias/offset b in H(omega); see Eq. (12)/(113)).
    paulis.append(krons([I] * n))
    
    return paulis

# make_training_states: the input states rho_1..rho_M (Paper Eq. (110)) used
#   for training. Paper Sec. VI.D.1 lists exactly this mix: computational-basis
#   states (|0>,|1>), Hadamard/+- basis, a GHZ state, the maximally mixed state
#   I/2^n, and a few random mixed states. Diverse states are needed so the
#   quantum (non-commuting) structure is observable.
def make_training_states(n):
    states = []
    dim = 2**n
    k0, k1 = np.array([1, 0]), np.array([0, 1])
    kp, km = np.array([1, 1])/np.sqrt(2), np.array([1, -1])/np.sqrt(2)

    # Computational basis states
    for bits in itertools.product([k0, k1], repeat=n):
        states.append(to_density(krons(bits)))
    
    # +/- basis states
    for bits in itertools.product([kp, km], repeat=n):
        states.append(to_density(krons(bits)))

    # GHZ state: (|00...0> + |11...1>) / sqrt(2)
    ghz_0 = krons([k0] * n)
    ghz_1 = krons([k1] * n)
    ghz = (ghz_0 + ghz_1) / np.sqrt(2)
    states.append(to_density(ghz))
    
    # Maximally mixed state
    states.append(np.eye(dim, dtype=complex) / dim)
    
    # Random mixed states
    for _ in range(3):
        A = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
        rho = A @ A.conj().T
        states.append(rho / np.trace(rho))

    return np.array(states)

# fdd_tanh_matrix: the matrix of "divided differences" of tanh,
#   F_lk = (tanh(l)-tanh(k))/(l-k), with the diagonal l==k replaced by the
#   derivative sech^2(l). This is the closed-form derivative of the matrix
#   function tanh(H) used in Theorem 1 / Appendix A.1 (derivative of the
#   matrix hyperbolic tangent) to get the gradient of the objective Eq. (20).
def fdd_tanh_matrix(eigenvalues):
    l = eigenvalues.reshape(-1, 1)
    k = eigenvalues.reshape(1, -1)
    diff = l - k
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = (np.tanh(l) - np.tanh(k)) / diff
    derivative = 1.0 / (np.cosh(l)**2)
    
    mask = np.abs(diff) < 1e-10
    res = np.where(mask, derivative, res)
    
    return res

# dfj: partial derivative d/d(omega_j) Tr[ g_T(H(omega)) rho ] for the tanh
#   neuron. Implements Theorem 1 / Eq. (20) (Sec. II.B): rotate H_j and rho
#   into the eigenbasis of H(omega), weight by the divided-difference matrix F,
#   and sum. This single gradient component feeds the training update Eq. (119).
def dfj(rho, eigvals, eigvecs, H_j_basis, T):
    H_j_tilde = eigvecs.T.conj() @ (H_j_basis / T) @ eigvecs
    rho_tilde = eigvecs.T.conj() @ rho @ eigvecs
    F = fdd_tanh_matrix(eigvals)
    return np.real(np.sum(F * H_j_tilde * rho_tilde.T))

In [ ]:
# optimize: the full training protocol of Paper Sec. VI.C, applied to
#   squared-loss / function approximation (Sec. VI.D.1, Fig. 7). It trains a
#   quantum model (TFIM) and a classical model (IM) in parallel against the
#   same target function and compares their loss curves.
def optimize(n=3):
    np.set_printoptions(suppress=True, precision=5)
    metrics_log = []
    
    # Generate Hamiltonians
    pauli_q = generate_paulis(n, model="quantum")
    pauli_c = generate_paulis(n, model="classical")

    training_states = make_training_states(n)
    N_states = len(training_states)
    T = 2.0   # temperature T (Paper: g_T(x)=tanh(x/T), Eq. (18)). Protocol step 1
              #   (Sec. VI.C) fixes T=2 and learning rate eta=1/10.
    
    # compute target Hamiltonian and function outputs
    # Data generation (Paper Sec. VI.A, Eq. (110)-(111)): pick a random target
    #   Hamiltonian H* (here a random TFIM), then label each state by the true
    #   function value f(rho)=Tr[tanh(H*/T) rho] (Eq. (120)). For squared-loss /
    #   regression the labels z_m=Tr[g_T(H*)rho_m] are kept as real numbers.
    target_p = (np.random.random(len(pauli_q)) - 0.5) * 4
    H_target = sum(p * mat for p, mat in zip(target_p, pauli_q)) 
    fs = np.array([np.real(np.trace(tanhm(H_target / T) @ rho)) for rho in training_states])

    # Protocol step 2 (Sec. VI.C): initialize the parameter vectors omega^q and
    #   omega^c of the quantum (H_Q) and classical (H_C) models at random.
    est_q = (np.random.random(len(pauli_q)) - 0.5)
    est_c = (np.random.random(len(pauli_c)) - 0.5)
    
    epochs = 2000  
    eta = 0.1       
    
    v_q = np.zeros(len(pauli_q))
    v_c = np.zeros(len(pauli_c))

    history_q = []
    history_c = []

    print(f"--- Running Optimization for {n} Qubits ---")
    print(f"{'Epoch':<10} | {'Quantum Loss':<15} | {'Classical Loss':<15}")
    print("-" * 45)
    
    for epoch in range(epochs):
        # --- Quantum Pass ---
        # --- Forward pass (Paper Eq. (117), protocol step 3): build H_Q(omega^q),
        #   diagonalize it, apply the activation tanh to its eigenvalues to form
        #   the activation observable g_T(H_Q)=tanh(H_Q/T) (Eq. (18)), then read
        #   off each neuron output f_q = Tr[ g_T(H_Q) rho ] (the objective Eq. (17)).
        H_q = sum(p * mat for p, mat in zip(est_q, pauli_q)) / T
        eval_q, evec_q = np.linalg.eigh(H_q)
        m_tanh_q = evec_q @ np.diag(np.tanh(eval_q)) @ evec_q.T.conj()
        fq_q = np.array([np.real(np.trace(m_tanh_q @ rho)) for rho in training_states])
        
        # --- Gradient (protocol step 4). Chain rule on squared loss
        #   L = mean_i (f_q[i]-target[i])^2: outer factor 2*(f_q-target), inner
        #   factor dfj = d Tr[g_T(H)rho]/d omega_j from Theorem 1 / Eq. (20).
        grad_q = np.zeros(len(pauli_q))
        for j in range(len(pauli_q)):
            g_j = sum(2 * (fq_q[i] - fs[i]) * dfj(training_states[i], eval_q, evec_q, pauli_q[j], T) for i in range(N_states))
            grad_q[j] = g_j / N_states
            
        # Gradient-descent update omega <- omega - eta*grad (Paper Eq. (119)).
        v_q = eta * grad_q
        est_q -= v_q
        
        # Squared-loss value L_q this epoch (the quantity plotted in Fig. 7).
        l_q = np.mean(np.square(fq_q - fs))
        history_q.append(l_q)

        # --- Classical pass: identical procedure for the classical Ising model
        #   H_C (Eq. (114)). Because H_C is built only from commuting Z operators,
        #   tanh(H_C) reduces to a classical neuron (Paper Sec. II.A); comparing
        #   its loss curve to the quantum one is the point of Fig. 7. ---
        H_c = sum(p * mat for p, mat in zip(est_c, pauli_c)) / T
        eval_c, evec_c = np.linalg.eigh(H_c)
        m_tanh_c = evec_c @ np.diag(np.tanh(eval_c)) @ evec_c.T.conj()
        fq_c = np.array([np.real(np.trace(m_tanh_c @ rho)) for rho in training_states])
        
        grad_c = np.zeros(len(pauli_c))
        for j in range(len(pauli_c)):
            g_j = sum(2 * (fq_c[i] - fs[i]) * dfj(training_states[i], eval_c, evec_c, pauli_c[j], T) for i in range(N_states))
            grad_c[j] = g_j / N_states
            
        v_c = eta * grad_c
        est_c -= v_c

        l_c = np.mean(np.square(fq_c - fs))
        history_c.append(l_c)

        epoch_data = {
            "Epoch": epoch,
            "Quantum_Loss": l_q,
            "Classical_Loss": l_c,
        }

        if epoch % 20 == 0:
            print(f"{epoch:<10} | {l_q:<15.8f} | {l_c:<15.8f}")
        metrics_log.append(epoch_data)

    print("\n--- Final Results ---")
    df_metrics = pd.DataFrame(metrics_log)
    csv_filename = f"outputs/mse_tanh_{n}qubit_ising.csv"
    df_metrics.to_csv(csv_filename, index=False)
    
    print("Target Params: ", target_p)
    print("Estimated Q:   ", est_q)
    print("Estimated C:   ", est_c)
    return history_q, history_c;

In [ ]:
# Run the experiment for n qubits (Paper used n in {2,...,7}; Fig. 7 shows 2 and 7).
n = 7
history_q, history_c = optimize(n);

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import FuncFormatter, NullFormatter
# plot: reproduces a panel of Paper Fig. 7 -- quantum (TFIM) vs classical (IM)
#   squared-loss vs epoch on a log scale. The persistent quantum<classical gap
#   is the headline numerical result of Sec. VI.D.1.
def plot(history_q, history_c, n):
    plt.figure(figsize=(10, 6))
    plt.plot(history_c, label=r'Classical Model ($H_\text{IM}$)', color='red', linewidth=2, linestyle='--')
    plt.plot(history_q, label=r'Quantum Model ($H_\text{TFIM}$)', color='blue', linewidth=2)
    plt.xlabel('Epoch', fontsize=24)
    plt.ylabel('Squared Loss', fontsize=24)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)    
    plt.yscale('log')
    plt.gca().yaxis.set_minor_formatter(NullFormatter())
    plt.legend(fontsize=24)
    plt.title(f'{n} Qubits, TFIM, tanh', fontsize=24)
    plt.subplots_adjust(left=0.15, right=0.97, bottom=0.18, top=0.90)
    plt.savefig(f"plots/mse_tanh_{n}qubit_ising.pdf", format="pdf")
    plt.show()

In [ ]:
plot(history_q, history_c, n)

In [ ]:
n = 2 
csv_filename = f"plots/mse_tanh_{n}qubit_ising.csv"
df = pd.read_csv(csv_filename)

history_q = df['Quantum_Loss'].values
history_c = df['Classical_Loss'].values

plot(history_q, history_c, n)